# 08 SHAP Explainability

Global feature importance rankings and local individual employee decision drivers.

### 1. Global & Local SHAP Explainability

In [ ]:
import pandas as pd
import joblib
import shap
import matplotlib.pyplot as plt

# Load model pipeline
pipeline = joblib.load("../models/v1/attrition_pipeline.joblib")
feature_names = joblib.load("../models/v1/feature_names.joblib")

df = pd.read_csv("../data/processed/employee_attrition_processed.csv")
df['IncomePerYearAtCompany'] = df['MonthlyIncome'] / (df['YearsAtCompany'] + 1.0)
df['PromotionLagRatio'] = df['YearsSinceLastPromotion'] / (df['YearsInCurrentRole'] + 1.0)
df['TotalSatisfactionScore'] = df['JobSatisfaction'] + df['EnvironmentSatisfaction'] + df['RelationshipSatisfaction'] + df['WorkLifeBalance']
df['ExperienceRatio'] = df['YearsAtCompany'] / (df['TotalWorkingYears'] + 1.0)

X = df.drop(columns=['EmployeeID', 'Attrition'])
X_trans = pipeline.named_steps['preprocessor'].transform(X)
X_trans_df = pd.DataFrame(X_trans, columns=feature_names)

# Compute Tree SHAP
clf = pipeline.named_steps['classifier']
explainer = shap.TreeExplainer(clf)
shap_values = explainer.shap_values(X_trans_df)

print("Global SHAP values computed successfully!")
# Top 5 most influential global features
mean_abs_shap = pd.Series(np.abs(shap_values).mean(axis=0), index=feature_names).sort_values(ascending=False)
print("\nTop 10 Global Features Driving Attrition:")
print(mean_abs_shap.head(10))